# 02 — Feature Extraction & Visualisation

Demonstrates extraction of all four feature channels: mel-spectrogram, MFCC (13), delta-MFCC, and chromagram.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from datautils import SpectrogramUtils

plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Pick a sample audio file ──────────────────────────────────────────────
SAMPLE_FILE = next(Path('../data/ravdess').rglob('*.wav'), None)
if SAMPLE_FILE is None:
    SAMPLE_FILE = next(Path('../data/toronto').rglob('*.wav'), None)
if SAMPLE_FILE is None:
    raise FileNotFoundError('No .wav files found in data/ravdess or data/toronto')

print(f'Sample file: {SAMPLE_FILE}')
y, sr = librosa.load(str(SAMPLE_FILE), sr=22050, mono=True)

In [ ]:
# ── Extract all features via SpectrogramUtils ─────────────────────────────
audio_dict = {'audio': y, 'sr': sr, 'path': SAMPLE_FILE, 'source_type': 'path'}
result = SpectrogramUtils.load_spectrogram(
    audio_dict=audio_dict, new_sr=22050, n_fft=2048,
    hop_length=512, n_mels=40, n_mfcc=13, target_duration=3.0,
)

mel    = result['mel_spectrogram']   # (40, T)
mfcc   = result['mfcc']              # (13, T)
mfcc_d = result['mfcc_delta']        # (13, T)
chroma = result['chromagram']        # (12, T)

print('mel:     ', mel.shape)
print('mfcc:    ', mfcc.shape)
print('delta:   ', mfcc_d.shape)
print('chroma:  ', chroma.shape)

In [ ]:
# ── Side-by-side feature visualisation ────────────────────────────────────
hop = 512
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.3)

feature_data = [
    (mel,    'Mel-Spectrogram (dB)',  'mel',   'magma'),
    (mfcc,   'MFCC (13 coeff)',       None,    'coolwarm'),
    (mfcc_d, 'Delta-MFCC',            None,    'coolwarm'),
    (chroma, 'Chromagram (12 bins)',  'chroma', 'Greens'),
]

for idx, (data, title, y_axis, cmap) in enumerate(feature_data):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    img = librosa.display.specshow(
        data, sr=22050, hop_length=hop, x_axis='time',
        y_axis=y_axis, ax=ax, cmap=cmap
    )
    ax.set_title(title)
    fig.colorbar(img, ax=ax)

plt.suptitle('Feature Channels — one audio clip', fontsize=13, y=1.01)
plt.show()

In [ ]:
# ── Stacked 4-channel tensor (as used by the model) ───────────────────────
import torch
import torch.nn.functional as F

def resize_to(arr, h):
    t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(h, arr.shape[1]), mode='bilinear', align_corners=False)
    return t.squeeze().numpy()

N_MELS = 40
stacked = np.stack([
    mel,
    resize_to(mfcc,   N_MELS),
    resize_to(mfcc_d, N_MELS),
    resize_to(chroma, N_MELS),
], axis=0)  # (4, 40, T)

print('Stacked tensor shape:', stacked.shape)

fig, axes = plt.subplots(1, 4, figsize=(18, 3))
channel_titles = ['Mel', 'MFCC (resized)', 'Delta-MFCC (resized)', 'Chroma (resized)']
for ax, ch, title in zip(axes, stacked, channel_titles):
    im = ax.imshow(ch, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(title)
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Freq bins')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('4-Channel Input to CNN-LSTM Model', fontsize=12)
plt.tight_layout()
plt.show()